In [1]:
pip install streamlit pyngrok pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 53.9 MB/s eta 0:00:00


In [2]:
%%writefile app.py
import streamlit as st
import pandas as pd
import sqlite3
import datetime
import os

# --- Configuration ---
DB_FILE = 'study_habit.db'
ADMIN_EMAIL = "admin@school.com"
ADMIN_PASSWORD = "admin123"

# --- Database Functions ---

def get_db_connection():
    conn = sqlite3.connect(DB_FILE)
    return conn

def init_db():
    """Initialize database and load CSV data if tables are empty."""
    conn = get_db_connection()
    c = conn.cursor()

    # 1. Create Users Table (Students + Admin)
    c.execute('''CREATE TABLE IF NOT EXISTS users (
                    student_id INTEGER PRIMARY KEY AUTOINCREMENT,
                    name TEXT,
                    email TEXT UNIQUE,
                    username TEXT,
                    password TEXT,
                    role TEXT
                )''')

    # 2. Create Study Logs Table
    c.execute('''CREATE TABLE IF NOT EXISTS study_logs (
                    log_id INTEGER PRIMARY KEY AUTOINCREMENT,
                    student_id INTEGER,
                    date TEXT,
                    study_hour REAL,
                    subject TEXT,
                    method_used TEXT,
                    distraction_time REAL,
                    quiz_score INTEGER,
                    day_name TEXT,
                    FOREIGN KEY(student_id) REFERENCES users(student_id)
                )''')

    # 3. Check and Load CSV Data
    c.execute("SELECT count(*) FROM users")
    if c.fetchone()[0] == 0:
        # Create Default Admin
        c.execute("INSERT INTO users (name, email, username, password, role) VALUES (?, ?, ?, ?, ?)",
                  ('Administrator', ADMIN_EMAIL, 'admin', ADMIN_PASSWORD, 'admin'))

        # Load Students CSV
        if os.path.exists('students.csv'):
            try:
                df_students = pd.read_csv('students.csv')
                # Append to DB (ensure columns match DB schema)
                df_students.to_sql('users', conn, if_exists='append', index=False)
                print("Students data loaded.")
            except Exception as e:
                print(f"Error loading students.csv: {e}")

        # Load Study Logs CSV
        if os.path.exists('study_logs.csv'):
            try:
                df_logs = pd.read_csv('study_logs.csv')
                df_logs.to_sql('study_logs', conn, if_exists='append', index=False)
                print("Study logs loaded.")
            except Exception as e:
                print(f"Error loading study_logs.csv: {e}")

    conn.commit()
    conn.close()

# --- Auth Functions ---

def login_user(email, password):
    conn = get_db_connection()
    c = conn.cursor()
    c.execute("SELECT student_id, name, role FROM users WHERE email=? AND password=?", (email, password))
    user = c.fetchone()
    conn.close()
    return user

def register_student(name, email, username, password):
    conn = get_db_connection()
    c = conn.cursor()
    try:
        c.execute("INSERT INTO users (name, email, username, password, role) VALUES (?, ?, ?, ?, ?)",
                  (name, email, username, password, 'student'))
        conn.commit()
        return True
    except sqlite3.IntegrityError:
        return False
    finally:
        conn.close()

# --- Core Interfaces ---

def student_interface(user_id, user_name):
    st.header(f"Welcome, {user_name} 👋")

    tab1, tab2 = st.tabs(["📝 Log Study Data", "📅 My History"])

    # 1. Input Fields for Study Data
    with tab1:
        st.subheader("Add New Study Log")
        with st.form("study_log_form"):
            col1, col2 = st.columns(2)
            with col1:
                date = st.date_input("Date")
                subject = st.text_input("Subject (e.g., Math, Science)")
                method = st.selectbox("Method Used", ["Pomodoro", "Flashcards", "Notes Review", "Video Lecture", "Group Study"])
            with col2:
                duration = st.number_input("Study Hours", min_value=0.0, step=0.5)
                distraction = st.number_input("Distraction Time (min)", min_value=0.0, step=1.0)
                score = st.number_input("Quiz Score (%)", min_value=0, max_value=100)

            submitted = st.form_submit_button("Save Log")

            if submitted:
                day_name = date.strftime("%A")
                conn = get_db_connection()
                c = conn.cursor()
                c.execute('''INSERT INTO study_logs (student_id, date, study_hour, subject, method_used, distraction_time, quiz_score, day_name)
                             VALUES (?, ?, ?, ?, ?, ?, ?, ?)''',
                          (user_id, date, duration, subject, method, distraction, score, day_name))
                conn.commit()
                conn.close()
                st.success("Study session logged successfully!")

    # 2. View Data (Fetch from DB in real-time)
    with tab2:
        st.subheader("Your Study History")
        conn = get_db_connection()
        query = f"SELECT * FROM study_logs WHERE student_id = {user_id} ORDER BY date DESC"
        df = pd.read_sql(query, conn)
        conn.close()
        st.dataframe(df)

def admin_interface():
    st.header("🛡️ Admin Panel")
    st.info("Manage student data and perform CRUD operations.")

    tab1, tab2 = st.tabs(["👥 Manage Students", "📊 Manage Logs"])

    # Admin Tab 1: Manage Students
    with tab1:
        conn = get_db_connection()
        all_users = pd.read_sql("SELECT * FROM users WHERE role='student'", conn)
        st.dataframe(all_users)

        # Delete Operation
        with st.expander("Delete Student"):
            student_id_to_del = st.number_input("Enter Student ID to Delete", min_value=0, step=1)
            if st.button("Delete Student"):
                c = conn.cursor()
                c.execute("DELETE FROM users WHERE student_id=?", (student_id_to_del,))
                c.execute("DELETE FROM study_logs WHERE student_id=?", (student_id_to_del,)) # Cascade delete logs
                conn.commit()
                st.success(f"Student {student_id_to_del} deleted.")
                st.rerun()
        conn.close()

    # Admin Tab 2: Manage Logs
    with tab2:
        conn = get_db_connection()
        all_logs = pd.read_sql("SELECT * FROM study_logs", conn)
        st.dataframe(all_logs)

        # Delete Operation
        with st.expander("Delete Log Entry"):
            log_id_to_del = st.number_input("Enter Log ID to Delete", min_value=0, step=1)
            if st.button("Delete Log"):
                c = conn.cursor()
                c.execute("DELETE FROM study_logs WHERE log_id=?", (log_id_to_del,))
                conn.commit()
                st.success(f"Log {log_id_to_del} deleted.")
                st.rerun()
        conn.close()

# --- Main App Logic ---

def main():
    st.set_page_config(page_title="StudyTrack AI", layout="wide")

    # Initialize DB on first run
    init_db()

    # Session State management
    if 'user' not in st.session_state:
        st.session_state.user = None

    # Sidebar Navigation
    with st.sidebar:
        st.title("StudyTrack AI")
        if st.session_state.user:
            st.write(f"Logged in as: **{st.session_state.user[1]}**")
            if st.button("Logout"):
                st.session_state.user = None
                st.rerun()
        else:
            st.write("Please Login to continue.")

    # Main Area
    if st.session_state.user:
        user_id, name, role = st.session_state.user

        if role == 'student':
            student_interface(user_id, name)
        elif role == 'admin':
            admin_interface()
    else:
        # Authentication Page
        tab_login, tab_register, tab_admin = st.tabs(["Student Login", "Register Student", "Admin Login"])

        with tab_login:
            st.subheader("Student Login")
            email = st.text_input("Email")
            password = st.text_input("Password", type="password")
            if st.button("Login"):
                user = login_user(email, password)
                if user and user[2] == 'student':
                    st.session_state.user = user
                    st.rerun()
                else:
                    st.error("Invalid student credentials")

        with tab_register:
            st.subheader("New Student Registration")
            new_name = st.text_input("Full Name")
            new_email = st.text_input("Email Address")
            new_user = st.text_input("Username")
            new_pass = st.text_input("Create Password", type="password")
            if st.button("Register"):
                if register_student(new_name, new_email, new_user, new_pass):
                    st.success("Registration successful! Please login.")
                else:
                    st.error("Email already exists.")

        with tab_admin:
            st.subheader("Admin Secure Login")
            ad_email = st.text_input("Admin Email")
            ad_pass = st.text_input("Admin Password", type="password")
            if st.button("Admin Login"):
                user = login_user(ad_email, ad_pass)
                if user and user[2] == 'admin':
                    st.session_state.user = user
                    st.rerun()
                else:
                    st.error("Invalid admin credentials")

if __name__ == '__main__':
    main()

Writing app.py


In [3]:
import subprocess
import time
from pyngrok import ngrok
import os

# 1. Check if app.py exists
if not os.path.exists('app.py'):
    print("❌ Error: app.py not found! Please save the code above as app.py first.")
else:
    # 2. Set your authtoken
    !ngrok config add-authtoken "35TQlXycOUQkriMQweSrRhhmt0s_HSa5nA2xZhdmsL19yj4g"

    # 3. Kill previous processes
    ngrok.kill()

    # 4. Run Streamlit
    process = subprocess.Popen([
        'streamlit', 'run', 'app.py',
        '--server.port', '8501',
        '--server.address', '127.0.0.1',
        '--server.headless', 'true'
    ])

    print("⏳ Waiting for Streamlit to start...")
    time.sleep(5)

    # 5. Create the Tunnel
    try:
        public_url = ngrok.connect(8501, "http")
        print(f"🚀 Application is running! Access it here: {public_url}")
    except Exception as e:
        print(f"❌ Ngrok error: {e}")

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
⏳ Waiting for Streamlit to start...
🚀 Application is running! Access it here: NgrokTunnel: "https://losing-optically-wilhelmina.ngrok-free.dev" -> "http://localhost:8501"
